# Vector Analogy Visualization

Classic static word embeddings often encode relationships as directions. This notebook demonstrates examples like:

```text
king - man + woman ~= queen
```

The demo uses `gensim` pretrained GloVe vectors, then projects selected words to 2D with PCA for visualization.

## Setup

Install dependencies if needed:

```bash
python3 -m pip install gensim numpy plotly
```

The first model load downloads `glove-wiki-gigaword-100` through `gensim.downloader`. PCA is computed with NumPy SVD.

In [ ]:
import importlib.util

USE_TOY = True
required = {"numpy": "numpy", "plotly": "plotly"}
if not USE_TOY:
    required["gensim"] = "gensim"
missing = [pkg for pkg, module in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError("Install missing packages with: python3 -m pip install " + " ".join(missing))

print("dependencies ok")

In [ ]:
from pathlib import Path
import importlib
import sys

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "vector_analogy":
    PROJECT_DIR = Path("/home/kazumasa/projects/notebooks/vector_analogy")
sys.path.insert(0, str(PROJECT_DIR))

import vector_analogy_visualization as viz
viz = importlib.reload(viz)

MODEL_NAME = "glove-wiki-gigaword-100"
model = viz.ToyStaticVectors() if USE_TOY else viz.load_vectors(MODEL_NAME)
len(model)

## Nearest Words After Vector Arithmetic

`positive=[a, c], negative=[b]` computes `a - b + c` and returns nearby words.

In [ ]:
viz.print_analogy_table(model, viz.ANALOGIES, topn=10)

## Dot Product vs Cosine Similarity

Dot product compares vectors directly, so large vector norms can increase the score. Cosine similarity divides by both vector lengths and compares direction.

In [ ]:
viz.print_similarity_comparison(model, "king", "man", "woman", topn=10)

## More Similar Word Examples

This table compares direct word pairs. Similar pairs, relation pairs, opposites, and distant concepts are included so the cosine scores have contrast.

In [ ]:
viz.print_pair_similarity_table(model, viz.SIMILARITY_PAIRS)

## 2D Projection

PCA compresses the selected word vectors into two dimensions. Distances and directions are only approximate after projection, but it is useful for seeing broad relationships. The HTML figure also includes a similarity table for `king - man + woman`.

In [ ]:
words = viz.available_words(model, viz.CONTEXT_WORDS)
points = viz.build_projection(model, words)
similarity_rows = viz.analogy_similarity_scores(model, "king", "man", "woman", topn=10)
pair_rows = viz.pair_similarity_scores(model, viz.SIMILARITY_PAIRS)
fig = viz.create_figure(points, viz.ANALOGIES, similarity_rows=similarity_rows, pair_rows=pair_rows)
fig

## Export HTML

In [ ]:
output = PROJECT_DIR / "vector_analogy.html"
fig.write_html(output, include_plotlyjs="cdn")
output

## Try Your Own Analogy

In [ ]:
a, b, c = "paris", "france", "japan"
model.most_similar(positive=[a, c], negative=[b], topn=10)

## Offline Toy Mode

If `gensim` is unavailable in the local environment, the script can still generate a toy HTML demo:

```bash
python3 notebooks/vector_analogy/vector_analogy_visualization.py --toy
```

This mode uses hand-built vectors, so it is useful for checking the visualization but not for demonstrating real pretrained embeddings.